In [1]:
#import of needed packages, adding relevant cities and their coordinates into a dictionary, declaration of weather dictionary

import requests
import pandas as pd
import copy as cp
from datetime import datetime

cities_coordinates = {
    "Atlanta": {"latitude": 33.7490, "longitude": -84.3880},
    "Charlotte": {"latitude": 35.2271, "longitude": -80.8431},
    "Chicago": {"latitude": 41.8781, "longitude": -87.6298},
    "Cleveland": {"latitude": 41.4993, "longitude": -81.6944},
    "Copenhagen": {"latitude": 55.6761, "longitude": 12.5683},
    "Denver": {"latitude": 39.7392, "longitude": -104.9903},
    "Las Vegas": {"latitude": 36.1699, "longitude": -115.1398},
    "Minneapolis": {"latitude": 44.9778, "longitude": -93.2650},
    "New York": {"latitude": 40.7128, "longitude": -74.0060},
    "Philadelphia": {"latitude": 39.9526, "longitude": -75.1652},
    "Pittsburgh": {"latitude": 40.4406, "longitude": -79.9959},
    "Scottsdale": {"latitude": 33.4942, "longitude": -111.9261},
    "Seattle": {"latitude": 47.6062, "longitude": -122.3321},
    "St Louis": {"latitude": 38.6270, "longitude": -90.1994},
    "Manchester": {"latitude": 53.4808, "longitude": -2.2426},
    "Boston": {"latitude": 42.3601, "longitude": -71.0589},
    "San Diego": {"latitude": 32.7157, "longitude": -117.1611},
    "Salt Lake City": {"latitude": 40.7608, "longitude": -111.8910},
    "New Orleans": {"latitude": 29.9511, "longitude": -90.0715},
    "Santa Monica": {"latitude": 34.0195, "longitude": -118.4912},
    "Detroit": {"latitude": 42.3314, "longitude": -83.0458},
    "Sydney": {"latitude": -33.8688, "longitude": 151.2093},
    "London": {"latitude": 51.5074, "longitude": -0.1278},
    #"Zagreb": {"latitude": 45.8154, "longitude": 15.9657}, this part was added just to see if SCD2 logic works as expected for DIM_LOCATION
}

cities_weather={}

In [2]:
#looping through cities and for each of them extracting relevant weather data into a dictionary

for city, coordinates in cities_coordinates.items():
    lat = coordinates["latitude"]
    lon = coordinates["longitude"]

    #hourly temp and auto timezone, the goal later will be to filter out the data for 12:00 PM local timezone for each city
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&timezone=auto&hourly=temperature_2m,precipitation,relative_humidity_2m,precipitation_probability&forecast_days=1"
    
    try:
        response = requests.get(url)
        response.raise_for_status()  #checking for http errors
        
        cities_weather[city] = response.json()
        print(f"data fetched for {city}")
        
    except Exception as e:
        print(f"fetching failed for {city}: {e}")

print("extraction complete")

data fetched for Atlanta
data fetched for Charlotte
data fetched for Chicago
data fetched for Cleveland
data fetched for Copenhagen
data fetched for Denver
data fetched for Las Vegas
data fetched for Minneapolis
data fetched for New York
data fetched for Philadelphia
data fetched for Pittsburgh
data fetched for Scottsdale
data fetched for Seattle
data fetched for St Louis
data fetched for Manchester
data fetched for Boston
data fetched for San Diego
data fetched for Salt Lake City
data fetched for New Orleans
data fetched for Santa Monica
data fetched for Detroit
data fetched for Sydney
data fetched for London
extraction complete


In [3]:
#in this part we are filtering only the weather for 12PM for each city on a given day
#this part was done using for loops and lower level code on purpose
#depending on time of the day when this part is triggered, the weather date for each city may not be for the same date

cities_weather_filtered=cp.deepcopy(cities_weather)

for city, city_data in cities_weather_filtered.items():
    helper_hourly={}
    for key, value in city_data["hourly"].items():
        if key=="time":
            i=0
            for string in value:
                string_check=string.endswith("T12:00")
                if string_check == True:
                    break
                i=i+1
        helper_hourly[key]=value[i]
    #print(helper_hourly)
    city_data["weather12PM"]=helper_hourly
    del city_data["hourly"]

#print(cities_weather_filtered)
    

In [4]:
#in this part we are loading data into a dataframe and doing needed changes so that we are left with columns which can be loaded into a csv file

df_start=pd.DataFrame(cities_weather_filtered)
df_T=df_start.T #cities are columns so we need to transpose
df=df_T.reset_index() #reset index so that cities are a column and not an index
df=df.rename(columns={"index": "city", "generationtime_ms": "generation_time_ms","hourly_units":"units","weather12PM":"weather"})
df_j1=df.join(pd.json_normalize(df['units']).add_suffix('_unit')) #units is in json in one column so we want to change that into separate columns, also joining with the original
df_j1=df_j1.drop(columns=['units', 'weather']) #removing columns because they are not needed anymore
df_final=df_j1.join(pd.json_normalize(df['weather'])) #same thing about json for weather and joining



In [5]:
#in this part we are extracting data into a csv file which we will later use as a source for our staging table in the database

date_str=datetime.now().strftime('%Y%m%d')
filename=f"export_{date_str}.csv"

df_csv = df_final.replace({';':''}, regex=True) #just in case because we will use ; as a separator
#Warning could be solved using pd.set_option('future.no_silent_downcasting', True), but it's not used here

df_csv.to_csv(filename, header = True, index = False, sep=';') #if a file already exists data will be overwritten

C:\Users\Matej\AppData\Local\Temp\ipykernel_8276\1019733781.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_csv = df_final.replace({';':''}, regex=True) #just in case because we will use ; as a separator


In [6]:
df_final

,city,latitude,longitude,generation_time_ms,utc_offset_seconds,timezone,timezone_abbreviation,elevation,time_unit,temperature_2m_unit,precipitation_unit,relative_humidity_2m_unit,precipitation_probability_unit,time,temperature_2m,precipitation,relative_humidity_2m,precipitation_probability
0,Atlanta,33.759865,-84.39586,0.074267,-14400,America/New_York,GMT-4,327.0,iso8601,°C,mm,%,%,2026-07-12T12:00,30.9,0.0,49,8
1,Charlotte,35.216976,-80.83189,0.982761,-14400,America/New_York,GMT-4,254.0,iso8601,°C,mm,%,%,2026-07-12T12:00,30.3,0.0,51,7
2,Chicago,41.879498,-87.64974,0.059962,-18000,America/Chicago,GMT-5,241.0,iso8601,°C,mm,%,%,2026-07-12T12:00,25.2,0.0,74,0
3,Cleveland,41.502296,-81.709496,0.105262,-14400,America/New_York,GMT-4,202.0,iso8601,°C,mm,%,%,2026-07-12T12:00,26.4,0.0,59,0
4,Copenhagen,55.75,12.5,0.094295,7200,Europe/Copenhagen,GMT+2,10.0,iso8601,°C,mm,%,%,2026-07-12T12:00,25.7,0.0,47,0
5,Denver,39.746895,-104.987076,0.087142,-21600,America/Denver,GMT-6,1599.0,iso8601,°C,mm,%,%,2026-07-12T12:00,33.9,0.0,21,0
6,Las Vegas,36.16438,-115.14392,0.050187,-25200,America/Los_Angeles,GMT-7,620.0,iso8601,°C,mm,%,%,2026-07-12T12:00,41.2,0.0,13,1
7,Minneapolis,44.96949,-93.26296,0.076532,-18000,America/Chicago,GMT-5,253.0,iso8601,°C,mm,%,%,2026-07-12T12:00,31.8,0.0,52,1
8,New York,40.710335,-73.99308,0.073075,-14400,America/New_York,GMT-4,32.0,iso8601,°C,mm,%,%,2026-07-12T12:00,27.5,0.0,46,0
9,Philadelphia,39.96188,-75.15539,0.0453,-14400,America/New_York,GMT-4,32.0,iso8601,°C,mm,%,%,2026-07-12T12:00,25.5,0.0,65,0
